In [5]:
"""
导包
""" 


import os
from pathlib import Path
from pprint import pprint
from sdp.reader.reader_factory import ReaderFactory
from sdp.dataset.dataset_factory import DatasetFactory
from sdp.processor.processor_factory import ProcessorFactory
from sdp.processor.processors.HwProcessor import HwProcessor
from sdp.processor.processors.BfeeProcessor import BfeeProcessor
from sdp.processor.processors.WiproxProcessor import WiproxProcessor

In [6]:
"""
执行位置
"""


PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# data path
folder_path = PROJECT_ROOT / "data/widar_data"
# param for BfeeProcessor(widar or gait)
task_type = 'Activity Recognition'
final_fs = 1000
# param for Wi_prox_Processor
num_samples = 100000


# 遍历目录下的文件 调整逻辑获取需要的文件
folder_path = Path(folder_path)
if not folder_path.exists() or not folder_path.is_dir():
    raise ValueError(f"无效的文件夹路径: {folder_path}")
files = [f for f in folder_path.rglob("*") if f.is_file() and "truth" not in f.name]
if not files:
    raise IOError(f"文件夹 {folder_path} 中没有文件")

# 使用第一个文件确定主格式
sample_file = files[0]
reader = ReaderFactory.create_reader(str(sample_file))
sample_frame = reader.read_file(sample_file).frames[0]
processor = ProcessorFactory.get_processor(sample_frame)


print(f"检测到主文件格式: {type(reader).__name__}")

print(f"开始处理 {len(files)} 个文件...")

csi_data_list = []
# 处理所有文件
for file_path in files:
    try:
        csi_data = reader.read_file(str(file_path))
        csi_data_list.append(csi_data)
        
        print(f"√ 已处理: {file_path.name}")
    
    except Exception as e:
        print(f"× 处理失败 {file_path.name}: {str(e)}")

print(f"处理完成! 共处理 {len(files)} 个文件")

# 情况处理'
global res
if type(processor) == HwProcessor:
    res = list(processor.process(csi_data_list, folder_path=folder_path))
elif type(processor) == BfeeProcessor:
    res = processor.process(csi_data_list, folder_path=folder_path, task_type=task_type, final_fs=final_fs)
elif type(processor) == WiproxProcessor:
    res = processor.process(csi_data_list, folder_path=folder_path, num_samples=num_samples)
# pprint(res)
# 构造dataset
dataset = DatasetFactory.create_dataset(res, reader)

print("process dataset success")

[Info] user2-1-1-1-1-r1.dat: B_FEE records=1934
检测到主文件格式: BfeeReader
开始处理 4 个文件...
[Info] user2-1-1-1-1-r1.dat: B_FEE records=1934
√ 已处理: user2-1-1-1-1-r1.dat
[Info] user2-1-1-1-2-r1.dat: B_FEE records=1884
√ 已处理: user2-1-1-1-2-r1.dat
[Info] user2-1-1-1-2-r2.dat: B_FEE records=1892
√ 已处理: user2-1-1-1-2-r2.dat
[Info] user2-1-1-1-2-r3.dat: B_FEE records=1901
√ 已处理: user2-1-1-1-2-r3.dat
处理完成! 共处理 4 个文件
[Info] Loaded user2-1-1-1-1-r1.dat with 1934 records.
[Info] Loaded user2-1-1-1-2-r1.dat with 1884 records.
[Info] Loaded user2-1-1-1-2-r2.dat with 1892 records.
[Info] Loaded user2-1-1-1-2-r3.dat with 1901 records.
[Done] Found 4 valid files in F:\Repository\pyProjects\sdp_benchmark\data\widar_data.
File: user2-1-1-1-1-r1.dat, Record count: 1934
  [Doppler] compute => T=1994, rx_acnt=3
  [Doppler] STFT => Z_sel.shape=(1077, 1), spec_2d.shape=(2, 1077, 1)
File: user2-1-1-1-2-r1.dat, Record count: 1884
  [Doppler] compute => T=1908, rx_acnt=3
  [Doppler] STFT => Z_sel.shape=(1031, 1), spec_2